In [ ]:
import os
import sys
import gymnasium as gym
import numpy as np
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import MatchKickoffReset
from src.rl.reward_shapers import Stage2Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

In [ ]:
def make_sparring_env(
    red_players: int = 1,  # 1 RL Agent + (red_players - 1) Heuristic Bots
    blue_players: int = 1, # M Heuristic Bots
    max_steps: int = 600,  # 10s rounds (or terminated on goal/concede)
    time_limit: float = 180.0,
    score_limit: int = 3,
):
    def _init():
        red_coord = TeamHeuristicCoordinator(team="red")
        blue_coord = TeamHeuristicCoordinator(team="blue")
        roster = []

        # 1. Red Team Slot 0: Primary RL Agent (DummyRLController injected by HaxballGymEnv)
        roster.append(PlayerSlot(
            team="red",
            stats=PlayerStats(name="RL_Agent", accel=3200.0)
        ))

        # 2. Additional Red Teammates (if red_players > 1)
        for i in range(1, red_players):
            roster.append(PlayerSlot(
                team="red",
                stats=PlayerStats(name=f"Red_Bot_{i}", accel=3000.0),
                controller=HeuristicBotController(red_coord)
            ))

        # 3. Blue Opponents
        for j in range(blue_players):
            roster.append(PlayerSlot(
                team="blue",
                stats=PlayerStats(name=f"Blue_Bot_{j + 1}", accel=3000.0),
                controller=HeuristicBotController(blue_coord)
            ))

        match_cfg = MatchConfig(
            mode=ClassicMatchMode(
                time_limit=time_limit,
                score_limit=score_limit,
                kickoff_timeout=10.0
            ),
            roster=roster,
            time_limit=time_limit,
            score_limit=score_limit,
        )

        return HaxballGymEnv(
            match_config=match_cfg,
            reward_shaper=Stage2Reward(team="red"),
            reset_strategy=MatchKickoffReset(),
            max_steps=max_steps
        )
    return _init

In [ ]:
# --- Match Scaling Configuration ---
RED_COUNT = 1   # Set to 2 for 2v2, 3 for 3v3, etc.
BLUE_COUNT = 1  # Opponent bot squad size

NUM_ENVS = 16
OBS_DIM = 80

train_envs = gym.vector.AsyncVectorEnv(
    [make_sparring_env(red_players=RED_COUNT, blue_players=BLUE_COUNT) for _ in range(NUM_ENVS)]
)
eval_env = make_sparring_env(red_players=RED_COUNT, blue_players=BLUE_COUNT)()

# Initialize 80-dim ActorCritic
model = ActorCritic(obs_dim=OBS_DIM).to(device)

# --- Optional: Warm-Start from Stage 1 (Weight Transplant) ---
stage1_ckpt = "models/stage1/best_stage1.pt"
if os.path.exists(stage1_ckpt):
    ckpt = torch.load(stage1_ckpt, map_location=device, weights_only=False)
    state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    # Check if first layer needs expansion from 68 to 80
    old_weight = state_dict.get("shared.0.weight", None)
    if old_weight is not None and old_weight.shape[1] == 68:
        print("🔄 Transplanting 68-dim weights into 80-dim architecture...")
        with torch.no_grad():
            model.shared[0].weight.data[:, :68] = old_weight
            # Copy all remaining matching layers
            for key in state_dict:
                if key != "shared.0.weight" and key in model.state_dict():
                    model.state_dict()[key].copy_(state_dict[key])
        print("✅ Transferred shooting motor skills from Stage 1!")
    elif old_weight is not None and old_weight.shape[1] == 80:
        model.load_state_dict(state_dict)
        print("✅ Loaded matching 80-dim checkpoint directly.")

In [ ]:
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2",
    lr_initial=2.5e-4,
    lr_final=1e-5,
    ent_coef_initial=0.01,
    ent_coef_final=0.001,
)

train_envs.close()
eval_env.close()